# Extending Elastic Agent Builder with LlamaParse Extract

We combine **LlamaParse Extract** (schema-driven PDF extraction) with **Elastic Agent Builder**
(workflow tools + search) to process complex documents like the
[World Bank Global Economic Prospects (Jan 2026)](https://www.worldbank.org/en/publication/global-economic-prospects).

Pipeline: Define schema > Create extraction agent > Upload PDF > Workflow (extract + index) > Create agent > Ask questions.

## Setup

In [ ]:
%pip install elasticsearch==9.4 python-dotenv==1.0.1 pydantic==2.10.6 requests==2.32.3 -q

In [2]:
import os
import json
from dotenv import load_dotenv

load_dotenv()

ELASTICSEARCH_URL = os.getenv("ELASTICSEARCH_URL")
ELASTICSEARCH_API_KEY = os.getenv("ELASTICSEARCH_API_KEY")
LLAMA_CLOUD_API_KEY = os.getenv("LLAMA_CLOUD_API_KEY")
KIBANA_URL = os.getenv("KIBANA_URL")

## Define the Extraction Schema

Pydantic model that defines what fields to extract from the PDF.

In [4]:
from pydantic import BaseModel, Field


class EconomicReportSummary(BaseModel):
    report_title: str = Field(description="Full title of the report")
    publication_date: str = Field(
        description="Publication date in YYYY-MM format, e.g. '2026-01'"
    )
    frontier_market_gdp_share_pct: float = Field(
        description="Frontier markets' share of global GDP in 2025 as a percentage, "
        "extracted from Figure ES.A bar chart"
    )
    frontier_market_investment_growth_2020s_pct: float = Field(
        description="Average annual per capita investment growth for frontier markets "
        "in the early 2020s (2020-24), extracted from Figure ES.C bar chart, as a percentage"
    )
    executive_summary: str = Field(
        description="Concise summary of the report's main findings and conclusions"
    )
    key_vulnerabilities: str = Field(
        description="Main vulnerabilities and risks facing frontier markets, as a paragraph"
    )
    policy_recommendations: str = Field(
        description="Key policy recommendations for frontier market policymakers, as a paragraph"
    )


print(json.dumps(EconomicReportSummary.model_json_schema(), indent=2))

{
  "properties": {
    "report_title": {
      "description": "Full title of the report",
      "title": "Report Title",
      "type": "string"
    },
    "publication_date": {
      "description": "Publication date in YYYY-MM format, e.g. '2026-01'",
      "title": "Publication Date",
      "type": "string"
    },
    "frontier_market_gdp_share_pct": {
      "description": "Frontier markets' share of global GDP in 2025 as a percentage, extracted from Figure ES.A bar chart",
      "title": "Frontier Market Gdp Share Pct",
      "type": "number"
    },
    "frontier_market_investment_growth_2020s_pct": {
      "description": "Average annual per capita investment growth for frontier markets in the early 2020s (2020-24), extracted from Figure ES.C bar chart, as a percentage",
      "title": "Frontier Market Investment Growth 2020S Pct",
      "type": "number"
    },
    "executive_summary": {
      "description": "Concise summary of the report's main findings and conclusions",
      "tit

## Create the Extract Configuration

LlamaExtract v2 replaced the v1 "extraction agent" with **saved configurations**: a reusable parameter set that pairs your schema with the extraction tier. We fetch the `project_id` and create the configuration via the LlamaCloud REST API. Save the printed `PROJECT_ID` and `CONFIGURATION_ID` — you'll need them in the workflow YAML.

In [5]:
import requests

LLAMA_CLOUD_BASE_URL = "https://api.cloud.llamaindex.ai"
llama_headers = {
    "Authorization": f"Bearer {LLAMA_CLOUD_API_KEY}",
    "Content-Type": "application/json",
}

# Fetch the project ID (uses the first available project)
projects_response = requests.get(
    f"{LLAMA_CLOUD_BASE_URL}/api/v1/projects",
    headers=llama_headers,
)
projects_response.raise_for_status()
PROJECT_ID = projects_response.json()[0]["id"]
print(f"Project ID: {PROJECT_ID}")

# Create a saved Extract v2 configuration with our schema
config_response = requests.post(
    f"{LLAMA_CLOUD_BASE_URL}/api/v1/beta/configurations",
    headers=llama_headers,
    params={"project_id": PROJECT_ID},
    json={
        "name": "global-economic-extractor",
        "parameters": {
            "product_type": "extract_v2",
            "data_schema": EconomicReportSummary.model_json_schema(),
            "extraction_target": "per_doc",
            "tier": "agentic",
        },
    },
)
config_response.raise_for_status()
CONFIGURATION_ID = config_response.json()["id"]
print(f"Configuration ID: {CONFIGURATION_ID}")

Project ID: dd082804-5c34-484d-8fa0-44dc494975b9
Configuration ID: cfg-mjknxxv4z138nq0p75j6ltca2xo9


## Create the Elasticsearch Index

Create the index with mappings matching our extraction schema. The workflow will index documents here.

In [6]:
from elasticsearch import Elasticsearch

es = Elasticsearch(
    ELASTICSEARCH_URL,
    api_key=ELASTICSEARCH_API_KEY,
)

print(f"Connected to Elasticsearch: {es.info()}")

Connected to Elasticsearch: {'name': 'instance-0000000005', 'cluster_name': 'fd31ad5f77d841d69b622c17ed64b766', 'cluster_uuid': 'YxmDkf9DSwWpvcCLv98q8A', 'version': {'number': '9.4.1', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': '3c7c6027c5769d860d87448e2749f4c550a239da', 'build_date': '2026-05-08T10:08:29.383338563Z', 'build_snapshot': False, 'lucene_version': '10.4.0', 'minimum_wire_compatibility_version': '8.19.0', 'minimum_index_compatibility_version': '8.0.0'}, 'tagline': 'You Know, for Search'}


In [7]:
INDEX_NAME = "economic-reports"

mappings = {
    "mappings": {
        "properties": {
            "report_title": {"type": "keyword"},
            "publication_date": {"type": "date", "format": "yyyy-MM"},
            "frontier_market_gdp_share_pct": {"type": "float"},
            "frontier_market_investment_growth_2020s_pct": {"type": "float"},
            "executive_summary": {"type": "text"},
            "key_vulnerabilities": {"type": "text"},
            "policy_recommendations": {"type": "text"},
        }
    }
}

if es.indices.exists(index=INDEX_NAME):
    es.indices.delete(index=INDEX_NAME)

es.indices.create(index=INDEX_NAME, body=mappings)
print(f"Created index: {INDEX_NAME}")

Created index: economic-reports


## Workflow Tool for Elastic Agent Builder

1. **upload_pdf:** uploads the PDF from a public URL to LlamaCloud, returns a `dfl-...` file ID.
2. **create_extraction_job:** starts the v2 extraction job referencing the saved configuration.
3. **wait_for_extraction:** initial 15s pause before polling.
4. **poll_until_done:** `while` loop containing `poll_wait` (10s) and `poll_get` (HTTP GET); repeats while the status is not `SUCCESS`, up to 19 iterations (~3 min cap).
5. **index_extracted_data:** indexes the extracted fields into Elasticsearch.
6. **verify_document:** confirms the document was indexed.

**Note:** Replace `projectId` and `configurationId` in the `consts` block with the values printed by the previous cell, and set `llamaCloudApiKey` to your own LlamaCloud API key.

```yaml
name: LlamaParse Extract Economic Report Processor
description: >
  Uploads a PDF from a URL to LlamaCloud, runs Extract v2,
  and indexes the structured results into Elasticsearch.
enabled: true

inputs:
  - name: pdf_url
    type: string
    description: Public URL of the PDF to process
    required: true

consts:
  indexName: economic-reports
  llamaBaseUrl: https://api.cloud.llamaindex.ai
  projectId: <YOUR_PROJECT_ID>
  configurationId: <YOUR_CONFIGURATION_ID>
  documentId: global-economic-prospects-jan-2026
  llamaCloudApiKey: llx-YOUR-API-KEY-HERE

triggers:
  - type: manual

steps:
  # Upload PDF from URL to LlamaCloud
  - name: upload_pdf
    type: http
    with:
      url: "{{ consts.llamaBaseUrl }}/api/v1/files/upload_from_url"
      method: PUT
      headers:
        Authorization: "Bearer {{ consts.llamaCloudApiKey }}"
        Content-Type: application/json
      body: |
        {
          "url": "{{ inputs.pdf_url }}"
        }

  # Create the Extract v2 job using our saved configuration
  - name: create_extraction_job
    type: http
    with:
      url: "{{ consts.llamaBaseUrl }}/api/v2/extract?project_id={{ consts.projectId }}"
      method: POST
      headers:
        Authorization: "Bearer {{ consts.llamaCloudApiKey }}"
        Content-Type: application/json
      body: |
        {
          "file_input": "{{ steps.upload_pdf.output.data.id }}",
          "configuration_id": "{{ consts.configurationId }}"
        }

  - name: wait_for_extraction
    type: wait
    with:
      duration: "15s"

  # Poll every 10s until status is COMPLETED (max ~3 min)
  - name: poll_until_done
    type: while
    condition: 'not steps.poll_get.output.data.status : "COMPLETED"'
    max-iterations:
      limit: 19
      on-limit: fail
    steps:
      - name: poll_wait
        type: wait
        with:
          duration: "10s"
      - name: poll_get
        type: http
        with:
          url: "{{ consts.llamaBaseUrl }}/api/v2/extract/{{ steps.create_extraction_job.output.data.id }}?project_id={{ consts.projectId }}"
          method: GET
          headers:
            Authorization: "Bearer {{ consts.llamaCloudApiKey }}"
            Accept: application/json

  - name: index_extracted_data
    type: elasticsearch.index
    with:
      index: "{{ consts.indexName }}"
      id: "{{ consts.documentId }}"
      document:
        report_title: "{{ steps.poll_get.output.data.extract_result.report_title }}"
        publication_date: "{{ steps.poll_get.output.data.extract_result.publication_date }}"
        frontier_market_gdp_share_pct: "{{ steps.poll_get.output.data.extract_result.frontier_market_gdp_share_pct }}"
        frontier_market_investment_growth_2020s_pct: "{{ steps.poll_get.output.data.extract_result.frontier_market_investment_growth_2020s_pct }}"
        executive_summary: "{{ steps.poll_get.output.data.extract_result.executive_summary }}"
        key_vulnerabilities: "{{ steps.poll_get.output.data.extract_result.key_vulnerabilities }}"
        policy_recommendations: "{{ steps.poll_get.output.data.extract_result.policy_recommendations }}"
      refresh: wait_for

  - name: verify_document
    type: elasticsearch.search
    with:
      index: "{{ consts.indexName }}"
      query:
        term:
          _id: "{{ consts.documentId }}"
```

## Create the Agent via API

Create a workflow tool that calls the extraction workflow, and two ES|QL tools to query the indexed results.
Then create an agent that uses all three tools. We use the [Agent Builder Kibana APIs](https://www.elastic.co/docs/explore-analyze/ai-features/agent-builder/kibana-api).

**Note:** After creating the workflow, copy its ID from the Elastic UI and set it below as `WORKFLOW_ID`.

In [8]:
import requests

headers = {
    "Authorization": f"ApiKey {ELASTICSEARCH_API_KEY}",
    "kbn-xsrf": "true",
    "Content-Type": "application/json",
}

WORKFLOW_NAME = "LlamaParse Extract Economic Report Processor"

response = requests.get(
    f"{KIBANA_URL}/api/workflows",
    headers=headers,
    params={"size": 20, "page": 1},
)
response.raise_for_status()

workflows = response.json()["results"]
workflow = next(wf for wf in workflows if wf["name"] == WORKFLOW_NAME)
WORKFLOW_ID = workflow["id"]
print(f"Workflow ID: {WORKFLOW_ID}")

Workflow ID: workflow-a3b8a34b-0073-4a08-98a8-63ba79953be0


In [13]:
# Create the workflow tool
workflow_tool_payload = {
    "id": "run_llamaextract_workflow",
    "type": "workflow",
    "description": (
        "Triggers the LlamaParse Extract extraction workflow. "
        "Use this tool to extract structured data from a PDF URL and index it into Elasticsearch. "
        "Requires only the public URL of the PDF to process."
    ),
    "tags": ["llama-extract", "workflow"],
    "configuration": {
        "workflow_id": WORKFLOW_ID,
    },
}

response = requests.post(
    f"{KIBANA_URL}/api/agent_builder/tools",
    headers=headers,
    json=workflow_tool_payload,
)
print(f"Workflow tool created: {response.status_code}")
print(json.dumps(response.json(), indent=2))

Workflow tool created: 200
{
  "id": "run_llamaextract_workflow",
  "type": "workflow",
  "description": "Triggers the LlamaParse Extract extraction workflow. Use this tool to extract structured data from a PDF URL and index it into Elasticsearch. Requires only the public URL of the PDF to process.",
  "tags": [
    "llama-extract",
    "workflow"
  ],
  "configuration": {
    "workflow_id": "workflow-a3b8a34b-0073-4a08-98a8-63ba79953be0"
  },
  "readonly": false,
  "schema": {
    "type": "object",
    "properties": {
      "pdf_url": {
        "type": "string",
        "description": "Public URL of the PDF to process"
      }
    },
    "required": [
      "pdf_url"
    ],
    "description": "Parameters needed to execute the workflow"
  }
}


In [14]:
# Delete existing tools to allow re-runs
for tool_id in ["query_structured_indicators", "search_economic_narratives"]:
    requests.delete(f"{KIBANA_URL}/api/agent_builder/tools/{tool_id}", headers=headers)

# Create the structured indicators query tool
structured_query_payload = {
    "id": "query_structured_indicators",
    "type": "esql",
    "description": (
        "Query structured economic indicators using exact filters on numeric fields. "
        "Use this for questions like 'Which reports show frontier market GDP share below 5%?' "
        "or 'Show investment growth for reports published after 2025-01'."
    ),
    "tags": ["economic-data", "llama-extract"],
    "configuration": {
        "query": (
            "FROM economic-reports "
            "| WHERE frontier_market_gdp_share_pct <= ?max_gdp_share "
            "| KEEP report_title, publication_date, "
            "frontier_market_gdp_share_pct, frontier_market_investment_growth_2020s_pct "
            "| SORT publication_date DESC "
            "| LIMIT 10"
        ),
        "params": {
            "max_gdp_share": {
                "type": "float",
                "description": "Maximum frontier market GDP share percentage to filter by",
            }
        },
    },
}

response = requests.post(
    f"{KIBANA_URL}/api/agent_builder/tools",
    headers=headers,
    json=structured_query_payload,
)
print(f"Structured query tool: {response.status_code}")
print(response.json())

# Create the narrative text search tool
text_search_payload = {
    "id": "search_economic_narratives",
    "type": "esql",
    "description": (
        "Search narrative content in economic reports. "
        "Use this for open-ended questions like 'What are the main risks for frontier markets?' "
        "or 'What does the report recommend for policymakers?'."
    ),
    "tags": ["economic-data", "llama-extract"],
    "configuration": {
        "query": (
            "FROM economic-reports "
            "| WHERE MATCH(executive_summary, ?query) "
            "OR MATCH(key_vulnerabilities, ?query) "
            "OR MATCH(policy_recommendations, ?query) "
            "| KEEP report_title, executive_summary, key_vulnerabilities, policy_recommendations "
            "| LIMIT 5"
        ),
        "params": {
            "query": {
                "type": "string",
                "description": "The search query to find relevant narrative content",
            }
        },
    },
}

response = requests.post(
    f"{KIBANA_URL}/api/agent_builder/tools",
    headers=headers,
    json=text_search_payload,
)
print(f"Text search tool: {response.status_code}")
print(json.dumps(response.json(), indent=2))

Structured query tool: 200
{'id': 'query_structured_indicators', 'type': 'esql', 'description': "Query structured economic indicators using exact filters on numeric fields. Use this for questions like 'Which reports show frontier market GDP share below 5%?' or 'Show investment growth for reports published after 2025-01'.", 'tags': ['economic-data', 'llama-extract'], 'configuration': {'query': 'FROM economic-reports | WHERE frontier_market_gdp_share_pct <= ?max_gdp_share | KEEP report_title, publication_date, frontier_market_gdp_share_pct, frontier_market_investment_growth_2020s_pct | SORT publication_date DESC | LIMIT 10', 'params': {'max_gdp_share': {'type': 'float', 'description': 'Maximum frontier market GDP share percentage to filter by'}}}, 'readonly': False, 'schema': {'type': 'object', 'properties': {'max_gdp_share': {'type': 'number', 'description': 'Maximum frontier market GDP share percentage to filter by'}}, 'required': ['max_gdp_share'], 'description': 'Parameters needed to

In [15]:
# Delete existing agent to allow re-runs
requests.delete(
    f"{KIBANA_URL}/api/agent_builder/agents/economic-report-analyst", headers=headers
)

# Create the agent with all three tools
agent_payload = {
    "id": "economic-report-analyst",
    "name": "Economic Report Analyst",
    "description": "Extracts and analyzes economic reports from PDFs using LlamaParse Extract and Elasticsearch.",
    "labels": ["economics", "llama-extract"],
    "configuration": {
        "instructions": (
            "You are an economic research assistant. You have three tools:\n"
            "1. run_llamaextract_workflow: Use this FIRST to extract and index data from a PDF.\n"
            "2. query_structured_indicators: Use this for structured questions that filter on "
            "numeric fields like GDP share or investment growth.\n"
            "3. search_economic_narratives: Use this for open-ended questions about risks, "
            "vulnerabilities, or policy recommendations.\n"
            "When presenting data, use clear formatting with bullet points or tables. "
            "Always cite the report title and publication date."
        ),
        "tools": [
            {
                "tool_ids": [
                    "run_llamaextract_workflow",
                    "query_structured_indicators",
                    "search_economic_narratives",
                ]
            }
        ],
    },
}

response = requests.post(
    f"{KIBANA_URL}/api/agent_builder/agents",
    headers=headers,
    json=agent_payload,
)
print(f"Agent created: {response.status_code}")
print(json.dumps(response.json(), indent=2))

Agent created: 200
{
  "id": "economic-report-analyst",
  "type": "chat",
  "name": "Economic Report Analyst",
  "description": "Extracts and analyzes economic reports from PDFs using LlamaParse Extract and Elasticsearch.",
  "labels": [
    "economics",
    "llama-extract"
  ],
  "visibility": "public",
  "created_by": {
    "username": "1481987933"
  },
  "configuration": {
    "instructions": "You are an economic research assistant. You have three tools:\n1. run_llamaextract_workflow: Use this FIRST to extract and index data from a PDF.\n2. query_structured_indicators: Use this for structured questions that filter on numeric fields like GDP share or investment growth.\n3. search_economic_narratives: Use this for open-ended questions about risks, vulnerabilities, or policy recommendations.\nWhen presenting data, use clear formatting with bullet points or tables. Always cite the report title and publication date.",
    "tools": [
      {
        "tool_ids": [
          "run_llamaextra

In [17]:
PDF_URL = "https://raw.githubusercontent.com/Delacrobix/notebook-llamaindex-agent-builder/main/Markets-Executive-Summary.pdf"  # Replace with actual PDF URL

# Ask the agent to extract and analyze the report
converse_payload = {
    "agent_id": "economic-report-analyst",
    "input": (
        f"Process this PDF and extract its data: {PDF_URL}\n\n"
        "Once extracted, answer: What are the main vulnerabilities facing frontier markets "
        "and what policy recommendations does the report suggest to address them?"
    ),
}

response = requests.post(
    f"{KIBANA_URL}/api/agent_builder/converse",
    headers=headers,
    json=converse_payload,
)
print(json.dumps(response.json(), indent=2))

{
  "conversation_id": "eefb33bd-563e-4a88-84b8-ac930a0c8065",
  "round_id": "174b7a65-71f9-4585-9e91-5d449182726b",
  "model_usage": {
    "connector_id": "Google-Gemini-2-5-Flash",
    "llm_calls": 4,
    "output_tokens": 576,
    "model": "google-gemini-2.5-flash",
    "input_tokens": 13130
  },
  "started_at": "2026-05-14T17:11:57.411Z",
  "time_to_last_token": 52630,
  "steps": [
    {
      "tool_call_group_id": "020dfcc8-e32d-4b92-857e-23b56386b2f1",
      "type": "reasoning",
      "reasoning": "The user wants to process a PDF and extract its data. The `run_llamaextract_workflow` tool is designed for this purpose, taking a PDF URL as input.",
      "tool_call_id": "tool_run_llamaextract_workflow_8LcDBClX24lpMzG1rEa7"
    },
    {
      "tool_call_group_id": "020dfcc8-e32d-4b92-857e-23b56386b2f1",
      "tool_call_id": "tool_run_llamaextract_workflow_8LcDBClX24lpMzG1rEa7",
      "tool_id": "run_llamaextract_workflow",
      "progression": [],
      "type": "tool_call",
      "pa

## Cleanup

In [ ]:
# Delete the Elasticsearch index
es.indices.delete(index=INDEX_NAME)

In [12]:
# Delete Agent Builder resources
requests.delete(
    f"{KIBANA_URL}/api/agent_builder/agents/economic-report-analyst", headers=headers
)
requests.delete(
    f"{KIBANA_URL}/api/agent_builder/tools/run_llamaextract_workflow", headers=headers
)
requests.delete(
    f"{KIBANA_URL}/api/agent_builder/tools/query_structured_indicators", headers=headers
)
requests.delete(
    f"{KIBANA_URL}/api/agent_builder/tools/search_economic_narratives", headers=headers
)

<Response [404]>